# Retail Sales Data Analysis

## 02 - Data Cleaning & Preprocessing

This notebook focuses on preparing the raw dataset
for further analysis.

In [1]:
# Import libraries

import pandas as pd
import numpy as np

# Load raw dataset

df = pd.read_csv(
    "../data/raw/sample_superstore.csv",
    encoding="cp1252"
)

# Create a working copy

df_clean = df.copy()

print("Raw dataset shape:", df.shape)
print("Working dataset shape:", df_clean.shape)

Raw dataset shape: (9994, 21)
Working dataset shape: (9994, 21)


In [2]:
# Convert date columns to datetime

date_columns = ["Order Date", "Ship Date"]

for column in date_columns:
    df_clean[column] = pd.to_datetime(
        df_clean[column],
        format="%m/%d/%Y"
    )

print(df_clean[date_columns].dtypes)

Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object


In [3]:
# Convert Postal Code to string

df_clean["Postal Code"] = (
    df_clean["Postal Code"].astype("string")
)

print(df_clean["Postal Code"].dtype)

string


In [4]:
# Check updated data types

df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   str           
 2   Order Date     9994 non-null   datetime64[us]
 3   Ship Date      9994 non-null   datetime64[us]
 4   Ship Mode      9994 non-null   str           
 5   Customer ID    9994 non-null   str           
 6   Customer Name  9994 non-null   str           
 7   Segment        9994 non-null   str           
 8   Country        9994 non-null   str           
 9   City           9994 non-null   str           
 10  State          9994 non-null   str           
 11  Postal Code    9994 non-null   string        
 12  Region         9994 non-null   str           
 13  Product ID     9994 non-null   str           
 14  Category       9994 non-null   str           
 15  Sub-Category   9994 non-null   s

In [5]:
# Preview cleaned dataset

display(df_clean.head())

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## 2. Data Consistency Check

This section checks the consistency and validity
of important variables in the dataset.

In [6]:
# Check data consistency

consistency_checks = {
    "Duplicate Row ID": df_clean["Row ID"].duplicated().sum(),

    "Ship Date Before Order Date": (
        df_clean["Ship Date"] < df_clean["Order Date"]
    ).sum(),

    "Non-positive Sales": (
        df_clean["Sales"] <= 0
    ).sum(),

    "Non-positive Quantity": (
        df_clean["Quantity"] <= 0
    ).sum(),

    "Invalid Discount": (
        ~df_clean["Discount"].between(0, 1)
    ).sum()
}

display(
    pd.Series(consistency_checks, name="Issue Count")
)

Duplicate Row ID               0
Ship Date Before Order Date    0
Non-positive Sales             0
Non-positive Quantity          0
Invalid Discount               0
Name: Issue Count, dtype: int64

## 3. Categorical Data Consistency

Check categorical columns for inconsistent values,
including different labels and unexpected categories.

In [7]:
# Define categorical columns

categorical_columns = [
    "Ship Mode",
    "Segment",
    "Country",
    "Region",
    "Category",
    "Sub-Category"
]

# Display unique values

for column in categorical_columns:
    print(f"\n{column}")
    print(df_clean[column].value_counts())


Ship Mode
Ship Mode
Standard Class    5968
Second Class      1945
First Class       1538
Same Day           543
Name: count, dtype: int64

Segment
Segment
Consumer       5191
Corporate      3020
Home Office    1783
Name: count, dtype: int64

Country
Country
United States    9994
Name: count, dtype: int64

Region
Region
West       3203
East       2848
Central    2323
South      1620
Name: count, dtype: int64

Category
Category
Office Supplies    6026
Furniture          2121
Technology         1847
Name: count, dtype: int64

Sub-Category
Sub-Category
Binders        1523
Paper          1370
Furnishings     957
Phones          889
Storage         846
Art             796
Accessories     775
Chairs          617
Appliances      466
Labels          364
Tables          319
Envelopes       254
Bookcases       228
Fasteners       217
Supplies        190
Machines        115
Copiers          68
Name: count, dtype: int64


In [8]:
# Identify leading and trailing whitespace

text_columns = df_clean.select_dtypes(
    include=["object", "string"]
).columns

whitespace_counts = {}

for column in text_columns:
    values = df_clean[column].astype("string")

    whitespace_counts[column] = (
        values != values.str.strip()
    ).sum()

# Display results

display(
    pd.Series(
        whitespace_counts,
        name="Whitespace Count"
    )
)

Order ID          0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code       0
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name     16
Name: Whitespace Count, dtype: int64

In [9]:
# Identify product names with leading or trailing whitespace

product_names = df_clean["Product Name"].astype("string")

whitespace_mask = (
    product_names != product_names.str.strip()
)

# Display affected records

product_name_issues = df_clean.loc[
    whitespace_mask,
    ["Row ID", "Product ID", "Product Name"]
].copy()

# Use repr() to make whitespace visible

product_name_issues["Product Name (repr)"] = (
    product_name_issues["Product Name"].map(repr)
)

display(product_name_issues)

,Row ID,Product ID,Product Name,Product Name (repr)
808,809,TEC-AC-10004901,Kensington SlimBlade Notebook Wireless Mouse w...,'Kensington SlimBlade Notebook Wireless Mouse ...
889,890,TEC-AC-10000927,Anker Ultrathin Bluetooth Wireless Keyboard Al...,'Anker Ultrathin Bluetooth Wireless Keyboard A...
1019,1020,OFF-ST-10001526,Iceberg Mobile Mega Data/Printer Cart,'Iceberg Mobile Mega Data/Printer Cart '
2169,2170,TEC-AC-10000927,Anker Ultrathin Bluetooth Wireless Keyboard Al...,'Anker Ultrathin Bluetooth Wireless Keyboard A...
2641,2642,TEC-AC-10000927,Anker Ultrathin Bluetooth Wireless Keyboard Al...,'Anker Ultrathin Bluetooth Wireless Keyboard A...
2759,2760,OFF-ST-10001526,Iceberg Mobile Mega Data/Printer Cart,'Iceberg Mobile Mega Data/Printer Cart '
2839,2840,OFF-ST-10001526,Iceberg Mobile Mega Data/Printer Cart,'Iceberg Mobile Mega Data/Printer Cart '
3590,3591,OFF-ST-10001526,Iceberg Mobile Mega Data/Printer Cart,'Iceberg Mobile Mega Data/Printer Cart '
3893,3894,TEC-AC-10004901,Kensington SlimBlade Notebook Wireless Mouse w...,'Kensington SlimBlade Notebook Wireless Mouse ...
4900,4901,OFF-ST-10001526,Iceberg Mobile Mega Data/Printer Cart,'Iceberg Mobile Mega Data/Printer Cart '


In [10]:
# Remove trailing whitespace from Product Name

df_clean["Product Name"] = (
    df_clean["Product Name"].str.rstrip()
)

print("Trailing whitespace removed!")

Trailing whitespace removed!


In [11]:
# Verify trailing whitespace

remaining_whitespace = (
    df_clean["Product Name"]
    != df_clean["Product Name"].str.rstrip()
).sum()

print("Remaining trailing whitespace:", remaining_whitespace)

Remaining trailing whitespace: 0


### Product Name Whitespace Cleaning

During categorical data inspection, 16 records
were found to contain trailing whitespace in
the Product Name column.

The issue was corrected using the str.rstrip()
method.

After cleaning, no trailing whitespace remained.

## 4. Final Data Quality Validation

Perform a final validation to ensure that the
cleaned dataset meets the required quality checks.

In [12]:
# Final data quality validation

text_columns = df_clean.select_dtypes(
    include=["object", "string"]
).columns

final_checks = {
    "Missing Values": df_clean.isnull().sum().sum(),

    "Duplicate Rows": df_clean.duplicated().sum(),

    "Duplicate Row ID": df_clean["Row ID"].duplicated().sum(),

    "Invalid Date Sequence": (
        df_clean["Ship Date"] < df_clean["Order Date"]
    ).sum(),

    "Non-positive Sales": (
        df_clean["Sales"] <= 0
    ).sum(),

    "Non-positive Quantity": (
        df_clean["Quantity"] <= 0
    ).sum(),

    "Invalid Discount": (
        ~df_clean["Discount"].between(0, 1)
    ).sum(),

    "Remaining Whitespace": sum(
        (
            df_clean[column].astype("string")
            != df_clean[column].astype("string").str.strip()
        ).sum()
        for column in text_columns
    )
}

display(
    pd.Series(final_checks, name="Issue Count")
)

Missing Values           0
Duplicate Rows           0
Duplicate Row ID         0
Invalid Date Sequence    0
Non-positive Sales       0
Non-positive Quantity    0
Invalid Discount         0
Remaining Whitespace     0
Name: Issue Count, dtype: int64

In [13]:
# Save cleaned dataset

df_clean.to_csv(
    "../data/processed/superstore_cleaned.csv",
    index=False
)

print("Cleaned dataset successfully saved!")

Cleaned dataset successfully saved!
